In [3]:
import duckdb
con = duckdb.connect("data/warehouse.duckdb", read_only=True)

1. Le nombre total de pages par document, trié du plus grand au plus petit.

In [5]:
con.sql("""
    SELECT filename, n_pages FROM documents ORDER BY n_pages DESC
""").df()

,filename,n_pages
0,Guide_Systemes_industriels__Mesures_detaillees...,124
1,anssi-guide-systemes_industriels-methode_de_cl...,54
2,Recommandations_de_sécurité_pour_un_système_d_...,38
3,Guide_pour_une_formation_sur_la_cybersécurité_...,36
4,anssi-fondamentaux-securisation-acme-v1-1.pdf,34
5,anssi-fondamentaux-zero-trust-v1.0.pdf,34
6,anssi-fondamentaux-securisation-poste-multi-en...,26
7,Fiche_Durcissement-securitee_VF.pdf,21
8,Referentiel_exigences_prestataires_integration...,21
9,anssi-fondamentaux-securisation_infrastructure...,20


2. Les 10 pages qui contiennent le plus de caractères.

In [4]:
con.sql("""
    SELECT d.filename, p.page_number, p.n_chars
    FROM pages p
    JOIN documents d USING (doc_id)
    ORDER BY p.n_chars DESC
    LIMIT 10
""").df()

,filename,page_number,n_chars
0,Referentiel_exigences_prestataires_integration...,3,6213
1,Fiche_Architectures-securisees_Protection_VF.pdf,5,4729
2,Fiche_Administration_Protection_VF.pdf,16,4681
3,Fiche_Administration_Protection_VF.pdf,9,4634
4,Fiche_Gestion-des-identites_Protection_VF.pdf,5,4539
5,Fiche_Durcissement-securitee_VF.pdf,16,4469
6,Document_Analyse_de_risques_FR.pdf,8,4464
7,Fiche_Gestion-des-identites_Protection_VF.pdf,10,4449
8,Fiche_Durcissement-securitee_VF.pdf,17,4439
9,Fiche_Durcissement-securitee_VF.pdf,14,4385


3. Les documents dont plus de 10 % des pages sont quasi vides (n_chars < 50).

In [6]:
con.sql("""
    SELECT d.filename,
       COUNT(*) FILTER (WHERE p.n_chars < 50) AS nb_vides,
       COUNT(*) AS nb_pages,
       COUNT(*) FILTER (WHERE p.n_chars < 50) * 1.0 / COUNT(*) AS proportion
FROM pages p
JOIN documents d USING (doc_id)
GROUP BY d.filename
HAVING COUNT(*) FILTER (WHERE p.n_chars < 50) * 1.0 / COUNT(*) > 0.1
ORDER BY proportion DESC
""").df()

,filename,nb_vides,nb_pages,proportion
0,Document_Analyse_de_risques_FR.pdf,2,16,0.125


In [7]:
con.sql("""
    WITH stats AS (
    SELECT d.filename,
           COUNT(*) FILTER (WHERE p.n_chars < 50) AS nb_vides,
           COUNT(*) AS nb_pages
    FROM pages p
    JOIN documents d USING (doc_id)
    GROUP BY d.filename
)
SELECT filename, nb_vides, nb_pages, nb_vides * 1.0 / nb_pages AS proportion
FROM stats
WHERE nb_vides * 1.0 / nb_pages > 0.1
ORDER BY proportion DESC
""").df()

,filename,nb_vides,nb_pages,proportion
0,Document_Analyse_de_risques_FR.pdf,2,16,0.125
